In [1]:
import json
from matplotlib import pyplot as plt
import numpy as np
from mrtdb_user import MRTDB
from tqdm import tqdm
from pktest import *

In [2]:
def validate_asn(x):
    if 1 <= x <= 64495:
        return True
    if 131072 <= x <= 4199999999:
        return True
    return False
db = MRTDB()

sizeof(struct list_head) = 16
sizeof(struct rb_node) = 24
sizeof(mrtdb_list_t) = 296
sizeof(mrtdb_key_t) = 48
sizeof(mrtdb_value_t) = 32
maxlen of origin_asn = 17
maxlen of value_list[0] = 162
maxlen of value_list[1] = 162
maxlen of value_list[2] = 0
maxlen of value_list[3] = 0
maxlen of value_list[4] = 0
load key_prefix time = 4617 ms
size of key_prefix = 988300
maxlen of value_list[0] = 0
maxlen of value_list[1] = 7498
maxlen of value_list[2] = 12496
maxlen of value_list[3] = 302
maxlen of value_list[4] = 9451
load key_as time = 202 ms
size of key_as = 76086
load key_as_path time = 7597 ms
size of key_as_path = 3257379
load key_next_hop time = 0 ms
size of key_next_hop = 35
prefix_ipv4_cnt = 0
prefix_ipv6_cnt = 0
prefix_ipswen_cnt = 988300
route_ipv4_cnt = 0
route_ipv6_cnt = 0
route_ipswen_cnt = 25747630


In [3]:
asip = []
with open('asip.jsonl') as f:
    for line in f:
        asip.append(json.loads(line))

In [7]:
cnt = 0
for i in asip:
    if i[1] == -2:
        cnt += 1
cnt

5125

In [3]:
raw = []
with open('agg0310.jsonl', 'r') as f:
    for line in f:
        raw.append(json.loads(line))
len(raw)

519617

In [4]:
stats = {}
for i,p in raw:
    stats[(i[1], i[2],0)] = stats.get((i[1], i[2],0), 0) + (len(p))
    stats[(i[1], i[2],1)] = stats.get((i[1], i[2],1), 0) + (len([x for x in p if x != '*']))
stats

{('icmp', 'none', 0): 984029,
 ('icmp', 'none', 1): 775764,
 ('icmp', 'rr', 0): 740312,
 ('icmp', 'rr', 1): 639958,
 ('icmp', 'nop', 0): 812716,
 ('icmp', 'nop', 1): 702908,
 ('icmp', 'unkctl3', 0): 820672,
 ('icmp', 'unkctl3', 1): 710244,
 ('icmp', 'unkres3', 0): 820746,
 ('icmp', 'unkres3', 1): 710515,
 ('icmp', 'unkctl40', 0): 819957,
 ('icmp', 'unkctl40', 1): 710205,
 ('icmp', 'unkres40', 0): 818453,
 ('icmp', 'unkres40', 1): 709114}

In [4]:
def test_hops_match(a, b):
    n = min(len(a), len(b))
    for i in range(n):
        if a[i] == '*' or b[i] == '*':
            continue
        if a[i] != b[i]:
            return False
    return True

def test_hops_match2(a, b):
    aa = []
    for i in a:
        n = db.ip2asn(i)[0]
        if validate_asn(n):
            if not aa or aa[-1] != n:
                if len(aa) >= 2 and aa[-2] == n and aa[-1] < 0:
                    aa.pop()
                else:
                    aa.append(n)
        else:
            if not aa or aa[-1] > 0:
                aa.append(-1)
            else:
                aa[-1] -= 1
    bb = []
    for i in b:
        n = db.ip2asn(i)[0]
        if validate_asn(n):
            if not bb or bb[-1] != n:
                if len(bb) >= 2 and bb[-2] == n and bb[-1] < 0:
                    bb.pop()
                else:
                    bb.append(n)
        else:
            if not bb or bb[-1] > 0:
                bb.append(-1)
            else:
                bb[-1] -= 1
    if not aa or not bb:
        return (aa, bb), (-1, -1)
    mat = np.zeros((len(aa) + 1, len(bb) + 1))
    mat[0][0] = 1
    if aa[0] < 0:
        mat[1][0] = 1
    if bb[0] < 0:
        mat[0][1] = 1
    i1 = -1
    j1 = -1
    for i in range(len(aa)):
        for j in range(len(bb)):
            if aa[i] > 0 and bb[j] > 0:
                if aa[i] == bb[j] and (mat[i][j] > 0 or mat[i + 1][j] > 0 or mat[i][j + 1] > 0):
                    mat[i + 1][j + 1] = 1
            elif aa[i] > 0 and bb[j] < 0:
                if mat[i][j] > 0 or mat[i + 1][j] > 0:
                    mat[i + 1][j + 1] = -bb[j]
                elif mat[i][j + 1] > 1:
                    mat[i + 1][j + 1] = mat[i][j + 1] - 1
            elif aa[i] < 0 and bb[j] > 0:
                if mat[i][j] > 0 or mat[i][j + 1] > 0:
                    mat[i + 1][j + 1] = -aa[i]
                elif mat[i + 1][j] > 1:
                    mat[i + 1][j + 1] = mat[i + 1][j] - 1
            if mat[i + 1][j + 1] > 0:
                if j >= j1:
                    j1 = j
                    i1 = i
    # print(aa)
    # print(bb)
    # print(mat)
    # print(i1, j1)
    return (aa, bb), (i1, j1)

for i in range(20):
    print(test_hops_match2(raw[i*7][1], raw[i*7+1][1]))

(([-4, 4134, 3356, 209], [23724, 4134, 3356]), (2, 2))
(([23724, -1, 4134, 3303, 207482], [23724, 4134]), (2, 1))
(([-1, 23724, -1, 4134, 2914, 9498, 134322], [23724, 4134, 2914]), (4, 2))
(([23724, -1, 4134, 3356], [23724, 4134, 3356]), (3, 2))
(([-1, 23724, 4134, 6327, 15296], [-1, 23724, 4134, 6327, 15296]), (4, 4))
(([23724, 4134, 5511, -4, 39010], [23724, 4134]), (1, 1))
(([23724, -1, 4134, 6453, 57866, 211588], [23724, -2, 4134, 6453]), (3, 3))
(([-3, 4134, -1, 5400, 2856, 211389], [23724, 4134, -1, 5400, 2856, 211389]), (5, 5))
(([23724, -2, 4134, 2914, 49685], [23724, 4134, 2914, 8455, 49685]), (3, 2))
(([23724, -2, 4134, 2497, 10010, -1, 23642], [23724, 4134, 2497, 10010]), (5, 3))
(([-4, 4134, 3356], [23724, -2, 4134, 3356]), (2, 3))
(([23724, 4134, 28917, 28994, 198124], [-2, 4134, 28917, -1, 28994, 198124]), (4, 5))
(([-1, 23724, -1, 4134, 3356, 12552, -1, 51815, 60582], [-3, 4134, -1, 3356]), (4, 3))
(([23724, -2, 4134, 3257, 12042, 397652], [23724, 4134, 3257, 12042, 3976

In [9]:
stats = {}
for i in tqdm(path):
    for j,a in path[i]:
        for k,b in path[i]:
            if j[2] < k[2]:
                m = test_hops_match(a, b)
                if m:
                    stats[(j[2],k[2])] = stats.get((j[2],k[2]), 0) + 1
stats

100%|██████████| 74231/74231 [00:03<00:00, 22971.31it/s]


{('none', 'unkres3'): 833,
 ('rr', 'unkctl3'): 62392,
 ('rr', 'unkres3'): 63485,
 ('rr', 'unkctl40'): 61705,
 ('rr', 'unkres40'): 67803,
 ('nop', 'rr'): 63510,
 ('nop', 'unkctl3'): 67495,
 ('nop', 'unkres3'): 63539,
 ('nop', 'unkctl40'): 64363,
 ('nop', 'unkres40'): 63794,
 ('unkctl3', 'unkres3'): 62743,
 ('unkctl3', 'unkctl40'): 64358,
 ('unkctl3', 'unkres40'): 62830,
 ('unkres3', 'unkres40'): 64565,
 ('unkctl40', 'unkres3'): 62547,
 ('unkctl40', 'unkres40'): 62296,
 ('none', 'nop'): 903,
 ('none', 'unkctl3'): 920,
 ('none', 'unkres40'): 819,
 ('none', 'unkctl40'): 858,
 ('none', 'rr'): 909}

In [5]:
path = {}
for i,p in raw:
    path.setdefault(i[0], []).append((i,p))

mat = {}
for i in tqdm(path):
    a = path[i][0]
    assert a[0][2] == 'none'
    for k,b in path[i]:
        if k[2] != 'none':
            m = test_hops_match2(a[1], b)
            mat.setdefault(k[2], []).append((m,a[1],b))

100%|██████████| 74231/74231 [02:03<00:00, 599.74it/s]


In [13]:
reach = []
dest = []
last1 = []
last2 = []
last3 = []
for i in mat:
    for ((asa, asb), (asi, asj)), a, b in mat[i]:
        if not asa or not asb:
            continue
        for j in asb:
            if j > 0:
                reach.append(j)
        if asi == len(asa) - 1 and asj == len(asb) - 1:
            if asa[-1] != asb[-1]:
                print(mat[i])
                assert 0
            dest.append(asa[-1])
        else:
            if asa[asi] > 0 and asb[asj] > 0:
                last1.append(asb[asj])
            elif asa[asi] * asb[asj] < 0:
                last2.append(max(asa[asi], asb[asj]))
            else:
                last3.append(0)
            pass
print(f'reach {len(set(reach))} dest {len(set(dest))} reach+dest {len(set(reach + dest))}')
print(f'last1 {len(set(last1))} last2 {len(set(last2))} last3 {len(set(last3))} last1+last2 {len(set(last1 + last2))}')

In [8]:
missing = set()
reach = set()
for i in mat:
    for ((asa, asb), (asi, asj)), a, b in mat[i]:
        for j in asb:
            if j > 0:
                reach.add(j)
        if asi + 1 < len(asa) and asa[asi + 1] > 0:
            missing.add(asa[asi + 1])
print(len(missing), len(reach), len(reach.intersection(missing)))

18010 40658 11024


In [15]:
stats = {}
for i in mat:
    for m,a,b in mat[i]:
        is_match = test_hops_match(a, b)
        # is_match = test_hops_match(m[0][1], m[0][0])
        # is_match = m[1][1] == len(m[0][1]) - 1
        is_reach = m[0][0] and m[0][1] and m[0][0][-1] == m[0][1][-1]
        stats[i] = stats.get(i, 0) + 1
        if is_match:
            stats[i + 'm'] = stats.get(i + 'm', 0) + 1
        if is_reach:
            stats[i + 'r'] = stats.get(i + 'r', 0) + 1
        if is_match and is_reach:
            stats[i + 'mr'] = stats.get(i + 'mr', 0) + 1
for i in mat:
    print(i, 'match%', stats[i + 'm'] / stats[i], 'match% in reach', stats[i + 'mr'] / stats[i + 'r'])

rr match% 0.012245557785830717 match% in reach 0.007370529009289648
nop match% 0.01216472902156781 match% in reach 0.007183203666204839
unkctl3 match% 0.01239374385364605 match% in reach 0.007708223457341069
unkres3 match% 0.01122172677183387 match% in reach 0.006913049648265656
unkctl40 match% 0.01155851328959599 match% in reach 0.007482564404953132
unkres40 match% 0.011033126321887082 match% in reach 0.006877747029138857


: 

In [20]:
stats = {}
for i, p in tqdm(raw):
    for j in p:
        r = set()
        for k in db.ip2asn(j):
            if validate_asn(k):
                r.add(k)
        # stats.setdefault((i[1], i[2]), set()).update(r)
        stats.setdefault((i[1], i[2]), {})
        for k in r:
            stats[(i[1], i[2])].setdefault(k, 0)
            stats[(i[1], i[2])][k] += 1

100%|██████████| 519617/519617 [00:37<00:00, 13973.93it/s]


In [28]:
stats = {}
for i, p in raw:
    stats[(i[1], i[2], 0)] = stats.get((i[1], i[2], 0), 0) + len(p)
    stats[(i[1], i[2], 1)] = stats.get((i[1], i[2], 1), 0) + len([j for j in p if j != '*'])
stats

{('icmp', 'none', 0): 984029,
 ('icmp', 'none', 1): 775764,
 ('icmp', 'rr', 0): 740312,
 ('icmp', 'rr', 1): 639958,
 ('icmp', 'nop', 0): 812716,
 ('icmp', 'nop', 1): 702908,
 ('icmp', 'unkctl3', 0): 820672,
 ('icmp', 'unkctl3', 1): 710244,
 ('icmp', 'unkres3', 0): 820746,
 ('icmp', 'unkres3', 1): 710515,
 ('icmp', 'unkctl40', 0): 819957,
 ('icmp', 'unkctl40', 1): 710205,
 ('icmp', 'unkres40', 0): 818453,
 ('icmp', 'unkres40', 1): 709114}

In [35]:
775764/984029

0.7883548147463134

In [22]:
for j in tier1:
    for i in stats:
        print(i, j, stats[i].get(j,0))

('tcp', 'none') 6762 1993
('tcp', 'rr') 6762 3
('tcp', 'nop') 6762 6
('tcp', 'unkctl3') 6762 3
('tcp', 'unkres3') 6762 4
('tcp', 'unkctl40') 6762 2
('tcp', 'unkres40') 6762 3
('tcp', 'none') 12956 4707
('tcp', 'rr') 12956 1
('tcp', 'nop') 12956 1
('tcp', 'unkctl3') 12956 1
('tcp', 'unkres3') 12956 1
('tcp', 'unkctl40') 12956 1
('tcp', 'unkres40') 12956 0
('tcp', 'none') 2914 39540
('tcp', 'rr') 2914 3386
('tcp', 'nop') 2914 3359
('tcp', 'unkctl3') 2914 3375
('tcp', 'unkres3') 2914 3374
('tcp', 'unkctl40') 2914 2121
('tcp', 'unkres40') 2914 2124
('tcp', 'none') 3356 29117
('tcp', 'rr') 3356 5
('tcp', 'nop') 3356 13
('tcp', 'unkctl3') 3356 10
('tcp', 'unkres3') 3356 8
('tcp', 'unkctl40') 3356 8
('tcp', 'unkres40') 3356 6
('tcp', 'none') 6453 26622
('tcp', 'rr') 6453 0
('tcp', 'nop') 6453 0
('tcp', 'unkctl3') 6453 0
('tcp', 'unkres3') 6453 0
('tcp', 'unkctl40') 6453 0
('tcp', 'unkres40') 6453 0
('tcp', 'none') 701 70
('tcp', 'rr') 701 0
('tcp', 'nop') 701 0
('tcp', 'unkctl3') 701 0
('tcp'

In [14]:
any = set()
all = set()
for i in stats:
    if i[1] != 'none':
        any.update(stats[i])
    all.update(stats[i])
len(any), len(all)

(16482, 48067)

In [15]:
for i in stats:
    for j in stats:
        if i == j:
            print(i, len(stats[i]))
        else:
            cap = len(set(stats[i]).intersection(set(stats[j])))
            print(i, j, cap, cap / min(len(stats[i]), len(stats[j])))

('tcp', 'none') 44294
('tcp', 'none') ('tcp', 'rr') 8824 0.7272127905060162
('tcp', 'none') ('tcp', 'nop') 9884 0.7482776894541601
('tcp', 'none') ('tcp', 'unkctl3') 11056 0.7713129621878052
('tcp', 'none') ('tcp', 'unkres3') 9698 0.7446253071253072
('tcp', 'none') ('tcp', 'unkctl40') 6320 0.7490814270475288
('tcp', 'none') ('tcp', 'unkres40') 6358 0.748264093209368
('tcp', 'rr') ('tcp', 'none') 8824 0.7272127905060162
('tcp', 'rr') 12134
('tcp', 'rr') ('tcp', 'nop') 10780 0.8884127245755727
('tcp', 'rr') ('tcp', 'unkctl3') 11074 0.9126421625185429
('tcp', 'rr') ('tcp', 'unkres3') 10681 0.8802538322070216
('tcp', 'rr') ('tcp', 'unkctl40') 7171 0.8499466635059856
('tcp', 'rr') ('tcp', 'unkres40') 6944 0.8172296104507473
('tcp', 'nop') ('tcp', 'none') 9884 0.7482776894541601
('tcp', 'nop') ('tcp', 'rr') 10780 0.8884127245755727
('tcp', 'nop') 13209
('tcp', 'nop') ('tcp', 'unkctl3') 12173 0.9215686274509803
('tcp', 'nop') ('tcp', 'unkres3') 12188 0.9358108108108109
('tcp', 'nop') ('tcp', 

In [17]:
asp = []
for i in tqdm(raw):
    path = []
    for j in i[3]:
        asn = db.ip2asn(j)
        if not validate_asn(asn):
            continue
        if len(path) > 0 and path[-1] == asn:
            continue
        path.append(asn)
    asp.append((i[0], i[1], i[2], path))

100%|██████████| 358575/358575 [00:55<00:00, 6491.81it/s]


In [18]:
records = {}
for i in asp:
    key = i[0], i[1]
    if key not in records:
        records[key] = []
    records[key].append(i[3])

In [19]:
nodes = {}
links = {}
for i in records:
    a = set()
    b = set()
    c = 0
    for j in records[i]:
        c += len(j)
        a.update(j)
        last = None
        for k in j:
            if last != None and last != k and last != 0 and k != 0:
                b.add((last, k))
            last = k
    print(i, c, c / len(records[i]))
    nodes[i] = a
    links[i] = b

('icmp', 'none') 354884 4.948532385135606
('icmp', 'rr') 284029 3.9605242975667574
('icmp', 'nop') 298265 4.159032280554975
('icmp', 'sid') 310340 4.327407097538869
('icmp', 'unknown') 302473 4.217708986962282


In [35]:
for i in records:
    for j in records:
        if i == j:
            print(i, len(nodes[i]), len(links[i]))
        elif nodes[i].intersection(nodes[j]):
            print(i, j, len(nodes[i].intersection(nodes[j])) / len(nodes[i].union(nodes[j])), len(links[i].intersection(links[j])) / len(links[i].union(links[j])))

('icmp', 'none') 42243 59970
('icmp', 'none') ('icmp', 'rr') 0.4461473762908275 0.19798524965384723
('icmp', 'none') ('icmp', 'nop') 0.5905165390255825 0.24454312005135886
('icmp', 'none') ('icmp', 'sid') 0.6510229199080417 0.2769117531084882
('icmp', 'none') ('icmp', 'unknown') 0.6345076153935589 0.26134303654467594
('icmp', 'rr') ('icmp', 'none') 0.4461473762908275 0.19798524965384723
('icmp', 'rr') 19655 24821
('icmp', 'rr') ('icmp', 'nop') 0.5753056655760289 0.27539647577092513
('icmp', 'rr') ('icmp', 'sid') 0.6228965240373356 0.29877665591352803
('icmp', 'rr') ('icmp', 'unknown') 0.6112950077686955 0.2852769156056671
('icmp', 'nop') ('icmp', 'none') 0.5905165390255825 0.24454312005135886
('icmp', 'nop') ('icmp', 'rr') 0.5753056655760289 0.27539647577092513
('icmp', 'nop') 26084 33082
('icmp', 'nop') ('icmp', 'sid') 0.8233380903388536 0.364734674850439
('icmp', 'nop') ('icmp', 'unknown') 0.8044333333333333 0.34554973821989526
('icmp', 'sid') ('icmp', 'none') 0.6510229199080417 0.27

In [73]:
for i in asp:
    if i[2] == ip[22444]:
        print(i)

('icmp', 'none', '65.151.36.1', [749, 23724, 174, 399221])
('icmp', 'rr', '65.151.36.1', [749, 23724, 4134])
('icmp', 'nop', '65.151.36.1', [749, 23724, 4134])
('icmp', 'sid', '65.151.36.1', [749, 23724, 4134])
('icmp', 'unknown', '65.151.36.1', [749, 23724, 4134])


In [95]:
stats = {}
for i in asp:
    key = i[0], i[1]
    if key not in stats:
        stats[key] = 0
    if 6453	 in i[3]:
        stats[key] += 1
stats

{('icmp', 'none'): 4593,
 ('icmp', 'rr'): 3789,
 ('icmp', 'nop'): 3638,
 ('icmp', 'sid'): 3985,
 ('icmp', 'unknown'): 3709}